In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV
from sklearn.preprocessing import TargetEncoder,StandardScaler,OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, r2_score

In [3]:
df=pd.read_csv(r"D:\eda\data\data.csv")
df.head()

,Age,OVA,POT,Release Clause,Best Position,Club,Value,Attacking,Defending,Goalkeeping
0,33,93,93,138400000.0,RW,FC Barcelona,103500000.0,429,91,54
1,35,92,92,75900000.0,ST,Juventus,63000000.0,437,84,58
2,27,91,93,159400000.0,GK,Atlético Madrid,120000000.0,95,57,437
3,29,91,91,161000000.0,CAM,Manchester City,129000000.0,407,186,56
4,28,91,91,166500000.0,LW,Paris Saint-Germain,132000000.0,408,94,59


In [4]:
features_set1=["Age","OVA","POT","Release Clause","Best Position","Club","Attacking","Defending","Goalkeeping"]
features_set2=["Age","OVA",]
features_set3=["Age","POT","Best Position","OVA"]

In [5]:
features_list=[features_set1,features_set2,features_set3]

In [6]:
num_pipeline=Pipeline([
    ("scaler",StandardScaler())
])

cat_pipeline=Pipeline([
    ("dummies",OneHotEncoder())
])

In [7]:
models={
    "Linear Regression":LinearRegression(),
    "Decision Tree":DecisionTreeRegressor(),
    "RandomForest":RandomForestRegressor(),
    "XGB Boost":XGBRegressor()
}

In [8]:
params={
    "Linear Regression":{},
    "Decision Tree":{
            "max_depth": [5, 10, 15, None],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 5],
            "max_features": ["sqrt", "log2", None]},
    "RandomForest":{
        "n_estimators": [100, 200, 300,500],
           "max_depth": [10, 20, None],
           "min_samples_split": [2, 5],
           "min_samples_leaf": [1, 2],
           "max_features": ["sqrt", "log2"]},
    "XGB Boost":{
        "n_estimators": [100, 300, 500],
        "learning_rate": [0.01, 0.05, 0.1],
        "max_depth": [3, 5, 7],
        "subsample": [0.7, 0.85, 1.0],
        "colsample_bytree": [0.7, 0.85, 1.0]}
}

In [9]:
def preprocessing(df, feature_set, num_pipeline, cat_pipeline):
    x = df[feature_set].copy()
    y = df["Value"].copy()
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
    
    if "Club" in feature_set:
        target_encoder = TargetEncoder(random_state=42, target_type="continuous")
        x_train["Club"] = target_encoder.fit_transform(x_train[["Club"]], y_train)
        x_test["Club"] = target_encoder.transform(x_test[["Club"]])

    numeric_cols = x_train.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = x_train.select_dtypes(include=["object"]).columns.tolist()
        
    transformers = []
    if numeric_cols:
        transformers.append(("num_pipeline", num_pipeline, numeric_cols))
    if cat_cols:
        transformers.append(("cat_pipeline", cat_pipeline, cat_cols))

    preprocessor = ColumnTransformer(transformers,remainder='passthrough')
    x_train_processed = preprocessor.fit_transform(x_train)
    x_test_processed = preprocessor.transform(x_test)

    return x_train_processed, x_test_processed, y_train, y_test

In [10]:
def model_evaluating(x_train, x_test, y_train, y_test, models, params):
    results = {}

    for name, model in models.items():
        para = params[name]
        gs = GridSearchCV(model, para, cv=3, scoring="neg_mean_absolute_error", n_jobs=-1)
        gs.fit(x_train, y_train)

        best_model = gs.best_estimator_
        y_pred = best_model.predict(x_test)
        test_mae = mean_absolute_error(y_test, y_pred)
        test_r2 = r2_score(y_test, y_pred)
        results[name] = {
            "best_params": gs.best_params_,
            "test_r2": test_r2,
            "test_mae": test_mae,
            "fitted_model": best_model,
            "test mae": test_mae,
        }
        print(f"{name}: test R2 = {test_r2:.4f}, test mae = {test_mae}, best params = {gs.best_params_}")

    best_name = max(results, key=lambda k: results[k]["test_r2"])
    print(f"\nBest model: {best_name} with R2 = {results[best_name]['test_r2']:.4f}")

    return results, best_name

In [11]:
for i in features_list:
    print(i)
    x_train,x_test,y_train,y_test=preprocessing(df, i, num_pipeline, cat_pipeline)
    result,best_name=model_evaluating(x_train, x_test, y_train, y_test, models, params)
    print("-"*50)

['Age', 'OVA', 'POT', 'Release Clause', 'Best Position', 'Club', 'Attacking', 'Defending', 'Goalkeeping']


C:\Users\parth\anaconda3\envs\testenv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
C:\Users\parth\AppData\Local\Temp\ipykernel_14224\1299602249.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = x_train.select_dtypes(include=["object"]).columns.tolist()


Linear Regression: test R2 = 0.9487, test mae = 838031.4685135137, best params = {}
Decision Tree: test R2 = 0.9783, test mae = 219276.87038988408, best params = {'max_depth': None, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
RandomForest: test R2 = 0.9832, test mae = 235808.33118405667, best params = {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
XGB Boost: test R2 = 0.9880, test mae = 144872.33716721128, best params = {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 500, 'subsample': 0.85}

Best model: XGB Boost with R2 = 0.9880
--------------------------------------------------
['Age', 'OVA']
Linear Regression: test R2 = 0.3670, test mae = 3084491.0893425844, best params = {}
Decision Tree: test R2 = 0.9578, test mae = 367534.34560470586, best params = {'max_depth': None, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
RandomForest: test R2 = 

C:\Users\parth\AppData\Local\Temp\ipykernel_14224\1299602249.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = x_train.select_dtypes(include=["object"]).columns.tolist()


Linear Regression: test R2 = 0.3697, test mae = 3085508.2920778394, best params = {}
Decision Tree: test R2 = 0.9680, test mae = 204009.43992928974, best params = {'max_depth': None, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
RandomForest: test R2 = 0.9788, test mae = 196680.60583955105, best params = {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}
XGB Boost: test R2 = 0.9854, test mae = 158936.05212175634, best params = {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 500, 'subsample': 0.7}

Best model: XGB Boost with R2 = 0.9854
--------------------------------------------------


In [14]:
best_name

'XGB Boost'

### The first set has release clause which is almost identical to value and its causing leaking in model the best performing set is feature set 3 XGB Boost model.